In [5]:
import pandas as pd
from pathlib import Path

In [6]:
# ---- 0) read all sheets and combine -----------------------------------------
xlsx = "../Results/TabPFN Regression Performance (Cleaned) - Copy.xlsx"          # <— change path
sheet_dfs = pd.read_excel(xlsx, sheet_name=None)  # dict: {sheet_name: df}

In [7]:
# standardize column names just in case
def clean_cols(df):
    df = df.rename(columns=lambda c: str(c).strip().lower().replace(" ", "_"))
    return df

dfs = []
for dataset, df in sheet_dfs.items():
    df = clean_cols(df)
    df["dataset"] = dataset
    dfs.append(df)

combined = pd.concat(dfs, ignore_index=True)

In [8]:
# ---- 1) choose metrics -------------------------------------------------------
# use the columns you actually have (below matches what you pasted)
metrics_norm = ["norm_raw_mae","norm_raw_rmse","norm_raw_r2",
                "norm_pro_mae","norm_pro_rmse","norm_pro_r2"]
metrics_raw  = ["raw_mae","raw_rmse","raw_r2","pro_mae","pro_rmse","pro_r2"]
time_cols    = [c for c in combined.columns if "fit_time" in c or "fit_time" in c or "processed_fit_time" in c]
metrics = metrics_norm + metrics_raw

In [9]:
# ---- 2) aggregate (mean, std) per model -------------------------------------
summary = combined.groupby("models")[metrics + time_cols].agg(["mean","std"])

# flatten the MultiIndex columns:  (metric, stat) -> metric_stat
summary.columns = [f"{m}_{stat}" for m, stat in summary.columns]
summary = summary.reset_index()

In [10]:
# ---- 3) wins per metric (higher-is-better for any column containing 'r2', else lower) --
wins = {}
for m in metrics:
    if "r2" in m:
        winners = combined.loc[combined.groupby("dataset")[m].idxmax()]
    else:
        winners = combined.loc[combined.groupby("dataset")[m].idxmin()]
    wins[m + "_wins"] = winners["models"].value_counts()

wins_df = pd.DataFrame(wins).fillna(0).astype(int).reset_index().rename(columns={"index":"models"})

In [11]:
# ---- 4) format mean ± std text columns (for the table view) -----------------
def mean_std_col(df, base):
    m = f"{base}_mean"; s = f"{base}_std"
    return df[m].map(lambda x: f"{x:.3f}") + " ± " + df[s].map(lambda x: f"{x:.2f}")

formatted = summary[["models"]].copy()

# choose what to display (example: mean normalized + mean raw + time)
display_norm = ["norm_raw_rmse","norm_raw_r2","norm_raw_mae",
                "norm_pro_rmse","norm_pro_r2","norm_pro_mae"]
display_raw  = ["raw_rmse","raw_r2","raw_mae","pro_rmse","pro_r2","pro_mae"]

for col in display_norm + display_raw:
    formatted[col] = mean_std_col(summary, col)

# include time if present
for tcol in time_cols:
    # both numeric columns exist; also add a text "mean ± std" column
    formatted[tcol] = mean_std_col(summary, tcol)

In [12]:
# ---- 5) merge wins and order columns ----------------------------------------
table = formatted.merge(wins_df, on="models", how="left").fillna(0)

# Example ordering similar to the paper’s sections
ordered_cols = (
    ["models"] +
    ["norm_raw_rmse","norm_raw_r2","norm_raw_mae",
     "norm_pro_rmse","norm_pro_r2","norm_pro_mae"] +
    ["raw_rmse","raw_r2","raw_mae","pro_rmse","pro_r2","pro_mae"] +
    sorted([c for c in table.columns if c.endswith("_wins")]) +
    [c for c in table.columns if c in time_cols]
)
table = table[ordered_cols]

In [13]:
# ---- 6) (optional) sort by a key metric, e.g., best normalized RMSE ----------
# For sorting, use the numeric "_mean" column from `summary`
table = table.merge(
    summary[["models","norm_raw_rmse_mean"]],
    on="models", how="left"
).sort_values("norm_raw_rmse_mean").drop(columns=["norm_raw_rmse_mean"])

# ---- 7) save -----------------------------------------------------------------
out_csv = Path("aggregated_results_table.csv")
table.to_csv(out_csv, index=False)
print(f"Saved: {out_csv.resolve()}")

Saved: C:\Users\mebub_9a7jdi8\Desktop\Research Paper\Practical Research\Jupyter Notebook\aggregated_results_table.csv


In [14]:
cleaned = pd.read_csv("aggregated_results_table.csv")
cleaned.head(5)

,models,norm_raw_rmse,norm_raw_r2,norm_raw_mae,norm_pro_rmse,norm_pro_r2,norm_pro_mae,raw_rmse,raw_r2,raw_mae,...,norm_raw_r2_wins,norm_raw_rmse_wins,pro_mae_wins,pro_r2_wins,pro_rmse_wins,raw_mae_wins,raw_r2_wins,raw_rmse_wins,raw_fit_time,processed_fit_time
0,TabPFN,0.503 ± 0.26,0.998 ± 0.00,0.447 ± 0.27,0.503 ± 0.27,0.995 ± 0.00,0.448 ± 0.27,190764.716 ± 479087.59,0.857 ± 0.16,137696.231 ± 344676.83,...,4.0,5.0,3.0,3.0,3.0,2.0,5.0,5.0,0.362 ± 0.06,0.331 ± 0.05
1,AutoTabPFN,0.508 ± 0.28,0.987 ± 0.02,0.447 ± 0.28,0.502 ± 0.27,0.993 ± 0.01,0.451 ± 0.27,202523.870 ± 510321.87,0.850 ± 0.17,146508.178 ± 368112.14,...,0.0,0.0,2.0,2.0,2.0,3.0,0.0,0.0,192.868 ± 127.52,180.366 ± 116.19
2,AutoGluon,0.525 ± 0.29,0.971 ± 0.03,0.483 ± 0.30,0.527 ± 0.29,0.969 ± 0.04,0.484 ± 0.31,202094.871 ± 509039.36,0.839 ± 0.18,147691.375 ± 371006.00,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,69.867 ± 68.54,65.581 ± 64.11
3,XGBoost,0.548 ± 0.30,0.950 ± 0.05,0.509 ± 0.30,0.548 ± 0.30,0.950 ± 0.05,0.509 ± 0.30,213013.913 ± 535245.90,0.823 ± 0.19,159011.251 ± 398754.25,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.108 ± 0.05,0.068 ± 0.04
4,LightGBM,0.587 ± 0.24,0.953 ± 0.03,0.554 ± 0.25,0.590 ± 0.24,0.948 ± 0.04,0.558 ± 0.25,207502.022 ± 521941.01,0.822 ± 0.17,153525.511 ± 384654.26,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.098 ± 0.03,0.051 ± 0.02
